In [82]:
import scipy.io as sio
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score,f1_score,accuracy_score
from torch_geometric.data import InMemoryDataset, Data
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')
from modelUtils import *
class CitationDomainData(InMemoryDataset):
    """
    引文数据集加载器：acmv9,citationv1,dblpv7 originally from CDNE
    """
    def __init__(self,root,name,use_pca=True,pca_dim=1000,transform=None,pre_transform=None,pre_filter=None):
        self.name=name
        self.use_pca = use_pca
        self.dim = pca_dim
        super(CitationDomainData, self).__init__(root, transform, pre_transform, pre_filter)
        self.data, self.slices = torch.load(self.processed_paths[0])
    @property
    def raw_file_names(self):
        return [f'{self.name}.mat']
    @property
    def processed_file_names(self):
        if self.use_pca:
            return [f'data_pca_{self.dim}.pt']
        else:
            return ['data.pt']

    def feature_compression(self,features):
        """Preprcessing of features"""
        features = features.toarray()
        feat = sp.lil_matrix(PCA(n_components=self.dim, random_state=0).fit_transform(features))
        return feat.toarray()

    def process(self):
        net = sio.loadmat(self.raw_dir+"\\"+self.name+".mat")
        features, adj, labels = net['attrb'], net['network'], net['group']
        if not isinstance(features, sp.lil_matrix):
            features = sp.lil_matrix(features)
        # citation networks do not use PCA, but blog networks use
        if self.use_pca:
            features = self.feature_compression(features)
            features = torch.from_numpy(features).to(torch.float)
        else:
            features = torch.from_numpy(features.todense()).to(torch.float)
        if not isinstance(adj,sp.coo_matrix):
            adj = sp.coo_matrix(adj)
        # label is float: to support BCEWithLogits loss
        y = torch.from_numpy(np.array(labels)).to(torch.float)
        data_list = []
        graph = Data(x=features,edge_index=adj,y=y)
        # train-val-test split
        random_node_indices = np.random.permutation(y.shape[0])
        train_size = int(len(random_node_indices) * 0.7)
        val_size = int(len(random_node_indices) * 0.1)
        train_node_indices = random_node_indices[:train_size]
        val_node_indices = random_node_indices[train_size:train_size+val_size]
        test_node_indices = random_node_indices[train_size+val_size:]
        train_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        train_mask[train_node_indices] = 1
        val_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        val_mask[val_node_indices] = 1
        test_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        test_mask[test_node_indices] = 1
        graph.train_mask = train_mask
        graph.val_mask = val_mask
        graph.test_mask = test_mask
        if self.pre_transform is not None:
            graph = self.pre_transform(graph)
        data_list.append(graph)
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

class FE(nn.Module):
    def __init__(self,input_dim,hidden_dim,output_dim):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.h1 = hidden_dim
        super(FE,self).__init__()
        self.fc1 = nn.Linear(input_dim,self.h1)
        self.bn1 = nn.BatchNorm1d(self.h1)
        self.fc2 = nn.Linear(self.h1,self.output_dim)
        self.bn2 = nn.BatchNorm1d(self.output_dim)
        nn.init.kaiming_normal_(self.fc1.weight.data)
        nn.init.constant_(self.fc1.bias.data, val=0.1)
        nn.init.kaiming_normal_(self.fc2.weight.data)
        nn.init.constant_(self.fc2.bias.data, val=0.1)
        nn.init.normal_(self.bn1.weight.data, 1.0, 0.02)
        nn.init.constant_(self.bn1.bias.data, 0.1)
        nn.init.normal_(self.bn2.weight.data, 1.0, 0.02)
        nn.init.constant_(self.bn2.bias.data, 0.1)
    def forward(self,inp):
        feat1 =  F.dropout(F.relu(self.bn1(self.fc1(inp))),training=self.training)  # 128-dim feature
        feat2 = F.relu(self.bn2(self.fc2(feat1)))
        return feat1,feat2

class NodeClassifier(nn.Module):
    def __init__(self,input_dim,output_dim):
        self.input_dim = input_dim
        self.output_dim = output_dim
        super(NodeClassifier, self).__init__()
        self.fc = nn.Linear(self.input_dim, self.output_dim)
        nn.init.kaiming_normal_(self.fc.weight.data)
        nn.init.constant_(self.fc.bias.data, val=0.1)
    def forward(self, inp):
        logits = self.fc(inp)
        return logits

class GradReverse(torch.autograd.Function):
    rate = 0.0
    @staticmethod
    def forward(ctx, *args, **kwargs):
        return args[0].view_as(args[0])

    @staticmethod
    def backward(ctx, *grad_outputs):
        grad_output = grad_outputs[0].neg()*GradReverse.rate
        return grad_output, None

class GRL(nn.Module):
    @staticmethod
    def forward(inp):
        return GradReverse.apply(inp)

class DomainClassifier(nn.Module):
    def __init__(self,input_dim, output_dim=2):
        super(DomainClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, output_dim)
        #self.fc2 = nn.Linear(10,10)
        #self.fc3 = nn.Linear(10, output_dim)
        #self.bn1 = nn.BatchNorm1d(10)
        nn.init.kaiming_normal_(self.fc1.weight.data)
        nn.init.constant_(self.fc1.bias.data, val=0.1)
        #nn.init.kaiming_normal_(self.fc2.weight.data)
        #nn.init.constant_(self.fc2.bias.data, val=0.1)
        #nn.init.kaiming_normal_(self.fc3.weight.data)
        #nn.init.constant_(self.fc3.bias.data, val=0.1)
        #nn.init.normal_(self.bn1.weight.data, 1.0, 0.02)
        #nn.init.constant_(self.bn1.bias.data, 0.1)
    def forward(self,x):
        #feat = F.dropout(F.relu(self.fc1(x)),training=self.training)
        #feat = F.dropout(F.relu(self.fc2(feat)),training=self.training)
        logits = self.fc1(x)
        return logits

class DANN(nn.Module):
    def __init__(self,input_dim, feat_dim_1=512, feat_dim_2=128, label_dim=5,domain_dim=2):
        super(DANN, self).__init__()
        self.FE = FE(input_dim,feat_dim_1,feat_dim_2)
        self.clf = NodeClassifier(feat_dim_2,label_dim)
        self.grl = GRL()
        self.domain_clf = DomainClassifier(feat_dim_2,domain_dim)
    def forward(self,x):
        """x = vstack(src,tgt)/src/tgt"""
        _,feat = self.FE(x)
        logits = self.clf(feat)
        feat_grl = self.grl(feat)
        domain_logits = self.domain_clf(feat_grl)
        return feat, logits, domain_logits

def weight_init(m):
    classname = m.__class__.__name__
    if classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.1)
    elif classname.find('Linear') != -1:
        nn.init.kaiming_normal_(m.weight.data)
        if m.bias.data is not None:
            torch.nn.init.constant_(m.bias.data, val=0.1)

def f1_scores(y_pred, y_true):
    """ y_pred: prob  y_true 0/1 """
    def predict(y_tru, y_pre):
        top_k_list = y_tru.sum(1)
        prediction = []
        for i in range(y_tru.shape[0]):
            pred_i = torch.zeros(y_tru.shape[1])
            pred_i[torch.argsort(y_pre[i, :])[-int(top_k_list[i].item()):]] = 1
            prediction.append(torch.reshape(pred_i, (1, -1)))
        prediction = torch.cat(prediction, dim=0)
        return prediction.to(torch.int32)
    results = {}
    predictions = predict(y_true, y_pred)
    averages = ["micro", "macro"]
    for average in averages:
        results[average] = f1_score(y_true.cpu().numpy(), predictions.cpu().numpy(), average=average)
    return results


def test(inp,net,name='src'):
    net.eval()
    with torch.no_grad():
        GradReverse.rate = 1.
        test_inp = inp.x
        test_label = inp.y
        _,logits, domain_logits = net(test_inp)
    label_pred = F.sigmoid(logits)
    result = f1_scores(label_pred,test_label)
    domain_logits = torch.argmax(F.softmax(domain_logits),1)
    if name=="src":
        domain_labels = torch.zeros_like(domain_logits)
    else:
        domain_labels = torch.ones_like(domain_logits)
    return result, accuracy_score(domain_logits.cpu().numpy(), domain_labels.cpu().numpy())

def save_embeddings(net, src_inp, tgt_inp,src,tgt):
    x_s = src_inp.x
    x_t = tgt_inp.x
    net.eval()
    with torch.no_grad():
        z_s,_,_ = net(x_s)
        z_t,_,_ = net(x_t)
    torch.save(z_s, f"embedding/dann_{src}_{tgt}-{src}.pt")
    torch.save(z_t, f"embedding/dann_{src}_{tgt}-{tgt}.pt")

In [83]:
def train_test_loop(src_inp,tgt_inp,net,criterion,batch_size,init_lr):
    """
    比较好的组合：a-c:bs=100+lr=1e-3+lamda*0.001 / d-a bs缩小到32
    :param src_inp:
    :param tgt_inp:
    :param net:
    :param criterion:
    :param batch_size:
    :param init_lr:
    :return:
    """
    # clear model before each training
    net.zero_grad()
    # use all labeled source data to train
    src_train_data = src_inp.x
    src_train_label = src_inp.y
    src_train_num = src_train_data.shape[0]
    tgt_train_data = tgt_inp.x
    tgt_train_num = tgt_train_data.shape[0]
    time = 0
    running_clf_loss = 0.
    running_domain_loss = 0.
    minibatch_times = int(src_train_num/batch_size)+1
    best_micro = 0.
    for epoch in range(epoches):
        p = float(epoch) / epoches
        lr = init_lr / (1. + 10 * p) ** 0.75
        grl_lambda = 2. / (1. + np.exp(-10. * p)) - 1  # gradually change from 0 to 1
        optimizer = torch.optim.Adam(net.parameters(),lr)
        #optimizer = torch.optim.SGD(net.parameters(),lr, 0.9,weight_decay=1e-3/2)
        GradReverse.rate = grl_lambda*0.001
        # mini-batch
        for start in range(0,src_train_num,batch_size):
            net.train()
            end = min(src_train_num, start+batch_size)
            src_input = src_train_data[start:end,:]
            src_label = src_train_label[start:end,:]
            # 如果目标域还有数据
            if start<tgt_train_num-1:
                end_ = min(tgt_train_num, start+batch_size)
                tgt_input = tgt_train_data[start:end_,:]
                x_s_t = torch.vstack([src_input,tgt_input])
                domain_label = torch.tensor(np.vstack([np.tile([1.,0.],[src_input.shape[0],1]),np.tile([0.,1.],[tgt_input.shape[0],1])])).to(device)
            # 源域长度>目标域
            else:
                x_s_t = src_input
                domain_label = torch.tensor(np.tile([1.,0.],[src_input.shape[0],1])).to(device)
            _,logits, domain_logits = net(x_s_t)
            clf_loss = criterion['clf'](logits[0:src_input.shape[0],:],src_label)
            domain_loss = criterion['domain'](domain_logits,torch.argmax(domain_label,1))
            # 源域长度<目标域，这时目标域还有剩余数据，也要用到domain classifier的训练中
            if end==src_train_num and end_<tgt_train_num-1:
                remaining_tgt_input = tgt_train_data[end_:tgt_train_num,:]
                _,_, remaining_domain_logits = net(remaining_tgt_input)
                domain_loss += criterion['domain'](remaining_domain_logits, torch.ones(remaining_tgt_input.shape[0],dtype=torch.long).to(device))
            optimizer.zero_grad()
            loss = clf_loss+domain_loss
            loss.backward()
            optimizer.step()
            time += 1
            running_clf_loss += clf_loss.item()
            running_domain_loss += domain_loss.item()
            if time%display_times==0 and time>0:
                # 源域全部拿来训练，所以评价结果没有意义（~100%），过拟合明显
                train_result,src_d_acc = test(src_inp,net,'src')
                test_result, tgt_d_acc = test(tgt_inp,net,'tgt')
                print(f"EPOCH:{epoch},CLF Loss:{running_clf_loss/display_times}, domain Loss:{running_domain_loss/display_times},Step:{epoch*minibatch_times+time}, Source MacroF1:{train_result['macro']}, TrainMicroF1:{train_result['micro']},Domain ACC:{src_d_acc}，Target MacroF1:{test_result['macro']}, TestMicroF1:{test_result['micro']}， Domain ACC:{tgt_d_acc}")
                running_clf_loss = 0.
                running_domain_loss = 0.
                if test_result['micro']>best_micro:
                    best_micro = test_result['micro']
                    save_embeddings(net, src_inp, tgt_inp,'dblpv7','acmv9')
                    print(f"Saving embeddings with MicroF1:{best_micro}")

random.seed(200)
np.random.seed(200)
torch.manual_seed(200)
src = CitationDomainData("data/dblpv7",name="dblpv7",use_pca=False)
tgt = CitationDomainData("data/acmv9",name="acmv9",use_pca=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
epoches = 30
lr = 0.001
batch_size = 32
display_times = 50
clf_criterion = nn.BCEWithLogitsLoss()
d_criterion = nn.CrossEntropyLoss()
criterion = {'clf':clf_criterion, 'domain':d_criterion}
source_data = src[0]
source_data = source_data.to(device)
target_data = tgt[0]
target_data = target_data.to(device)
print(source_data.x.shape[0],target_data.x.shape[0])
net = DANN(input_dim=source_data.x.shape[1]).to(device)
train_test_loop(source_data, target_data, net, criterion,batch_size,init_lr=lr)

cuda
5484 9360
EPOCH:0,CLF Loss:0.5603035151958465, domain Loss:0.8386763226985932,Step:50, Source MacroF1:0.36745802886791123, TrainMicroF1:0.4559010729223495,Domain ACC:0.6905543398978847，Target MacroF1:0.18947933816496312, TestMicroF1:0.2683834624283271， Domain ACC:0.6672008547008547
EPOCH:0,CLF Loss:0.5067653733491898, domain Loss:2.65039381980896,Step:100, Source MacroF1:0.2789948924775768, TrainMicroF1:0.3586106564829969,Domain ACC:0.056528081692195475，Target MacroF1:0.15647529914743924, TestMicroF1:0.33125440096569764， Domain ACC:0.00844017094017094
EPOCH:0,CLF Loss:0.595697956085205, domain Loss:0.919515917301178,Step:150, Source MacroF1:0.30806768061836054, TrainMicroF1:0.44808146935806503,Domain ACC:0.9223194748358862，Target MacroF1:0.14283198142074338, TestMicroF1:0.31576300171008953， Domain ACC:0.8689102564102564
EPOCH:1,CLF Loss:0.5148651593923569, domain Loss:0.34628512024879454,Step:372, Source MacroF1:0.3594039511374971, TrainMicroF1:0.4980905619203492,Domain ACC:0.9387

KeyboardInterrupt: 

In [8]:
from modelUtils import MMD_loss,GaussianKernel,MultipleKernelMaximumMeanDiscrepancy
import matplotlib.pyplot as plt
import torch
datasets = ["acmv9","citationv1","dblpv7"]
for i in range(len(datasets)):
    for j in range(i+1,len(datasets)):
        dual_ds = [(i,j),(j,i)]
        for ds in dual_ds:
            src_name = datasets[ds[0]]
            tgt_name = datasets[ds[1]]
            print(src_name, tgt_name)
            src = CitationDomainData(f"data/{src_name}",name=src_name,use_pca=False)
            tgt = CitationDomainData(f"data/{tgt_name}",name=tgt_name,use_pca=False)
            source_data = src[0]
            source_data = source_data.to(device)
            target_data = tgt[0]
            target_data = target_data.to(device)
            size = min(source_data.x.shape[0],target_data.x.shape[0])
            random_indices = np.random.permutation(size)
            x = source_data.x[random_indices]
            y = target_data.x[random_indices]
            sigmas = [
                1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 5, 10, 15, 20, 25, 30, 35, 100,
                1e3, 1e4, 1e5, 1e6
            ]
            kernels = []
            for sigma in sigmas:
                kernels.append(GaussianKernel(sigma=sigma))
            attr_mmd = 0.
            mmd2 = MultipleKernelMaximumMeanDiscrepancy(kernels=kernels)
            bs = 64
            for start in range(0,size,bs):
                end = min(size-1,start+bs)
                attr_mmd += mmd2(x[start:end],y[start:end])
            print(src_name, tgt_name, attr_mmd)

acmv9 citationv1
acmv9 citationv1 tensor(6.4770, device='cuda:0')
citationv1 acmv9
citationv1 acmv9 tensor(6.5710, device='cuda:0')
acmv9 dblpv7
acmv9 dblpv7 tensor(6.4672, device='cuda:0')
dblpv7 acmv9
dblpv7 acmv9 tensor(6.3061, device='cuda:0')
citationv1 dblpv7
citationv1 dblpv7 tensor(5.3429, device='cuda:0')
dblpv7 citationv1
dblpv7 citationv1 tensor(5.3840, device='cuda:0')


In [71]:
x = torch.ones(100,1,dtype=torch.long)
x

tensor([[1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],

In [23]:
x = torch.rand(3,2)
y = F.softmax(x)
print(x, y)

tensor([[0.7705, 0.0695],
        [0.7919, 0.6532],
        [0.9776, 0.8765]]) tensor([[0.6684, 0.3316],
        [0.5346, 0.4654],
        [0.5253, 0.4747]])
